<div align="center" style="padding: 25px; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); border-radius: 12px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <h1 style="color: #ffffff; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin-bottom: 10px; font-size: 2.5em; text-shadow: 0 2px 4px rgba(0,0,0,0.2);">
        🔥 Mytho<br>
        <span style="font-size: 0.5em; font-weight: 400; color: #e0e0e0;">Recurrent-Depth Transformer</span>
    </h1>
    <p style="color: #bbdefb; font-size: 1.2em; font-weight: 500; letter-spacing: 0.5px;">🚀 T4 Pretraining Notebook &mdash; Fast Colab setup for 16 GB runs</p>
</div>

<table style="width: 100%; border: none; margin-bottom: 20px;">
  <tr style="border: none;">
    <td style="width: 50%; vertical-align: top; padding: 0 10px 0 0; border: none;">
        <div style="padding: 20px; background: #f8f9fa; border-top: 4px solid #2196f3; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); height: 100%;">
            <h3 style="margin-top: 0; color: #1565c0; font-size: 1.3em;">✨ Architecture Highlights</h3>
            <ul style="margin-bottom: 0; color: #444; line-height: 1.6;">
                <li><b>Recurrent depth</b> with Adaptive Computation Time (ACT)</li>
                <li><b>Multi-Latent Attention</b> (DeepSeek-V2 style KV compression)</li>
                <li><b>Mixture-of-Experts</b> with SwiGLU + load-balancing</li>
                <li><b>Latent scratchpad</b> for internal reasoning</li>
                <li><b>Verifier-guided halting</b></li>
            </ul>
        </div>
    </td>
    <td style="width: 50%; vertical-align: top; padding: 0 0 0 10px; border: none;">
        <div style="padding: 20px; background: #f8f9fa; border-top: 4px solid #ff9800; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); height: 100%;">
            <h3 style="margin-top: 0; color: #e65100; font-size: 1.3em;">⚡ T4-Friendly Defaults</h3>
            <table style="width: 100%; border-collapse: collapse; color: #444;">
                <tr style="border-bottom: 1px solid #e0e0e0;"><td style="padding: 8px 0;"><b>Precision</b></td><td style="padding: 8px 0;">FP16 AMP (no BF16 on T4)</td></tr>
                <tr style="border-bottom: 1px solid #e0e0e0;"><td style="padding: 8px 0;"><b><code>max_depth</code></b></td><td style="padding: 8px 0;">8 (from 12)</td></tr>
                <tr style="border-bottom: 1px solid #e0e0e0;"><td style="padding: 8px 0;"><b><code>seq_len</code></b></td><td style="padding: 8px 0;">512 (from 2048)</td></tr>
                <tr style="border-bottom: 1px solid #e0e0e0;"><td style="padding: 8px 0;"><b>Effective Batch</b></td><td style="padding: 8px 0;">32 (via grad accum)</td></tr>
                <tr><td style="padding: 8px 0;"><b>Optional</b></td><td style="padding: 8px 0;">Gradient checkpointing</td></tr>
            </table>
        </div>
    </td>
  </tr>
</table>

<div style="background-color: #fff3e0; padding: 15px 20px; border-radius: 8px; border-left: 5px solid #ff9800; color: #e65100; font-weight: 500; margin-top: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <span style="font-size: 1.2em; margin-right: 10px;">⚙️</span> <b>Colab Runtime:</b> <code>Runtime → Change runtime type → Hardware accelerator → T4 GPU</code> before running.
</div>

<div style="padding: 15px 20px; background-color: #F1F8E9; border-left: 6px solid #4CAF50; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #2E7D32; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">1️⃣</span> Environment Setup
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Verify GPU availability and install core dependencies.</p>
</div>

In [ ]:
# ── Verify GPU ───────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "❌ No GPU — go to Runtime → Change runtime type → T4 GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB")

In [ ]:
# ── Install dependencies ─────────────────────────────────────────
!pip install -q torch>=2.1 tiktoken>=0.5 datasets>=2.16 tqdm>=4.65 matplotlib
print("✅ Dependencies installed")

<div style="padding: 15px 20px; background-color: #E3F2FD; border-left: 6px solid #2196F3; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #1565C0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">2️⃣</span> Clone Repository
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Pull the latest Mytho source and move into the repo folder. Safe to re-run.</p>
</div>

In [ ]:
# ── Clone Mytho from GitHub ──────────────────────────────────────
import os

REPO_URL = "https://github.com/dev-760/Mytho.git"
REPO_DIR = "Mytho"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print(f"'{REPO_DIR}' already exists — pulling latest...")
    %cd $REPO_DIR
    !git pull
    %cd ..

# Change working directory into the repo
%cd $REPO_DIR
print(f"\n✅ Working directory: {os.getcwd()}")
!ls mytho_model/

In [ ]:
# ── Patch AMP API for newer PyTorch (safe to re-run) ──────────────
from pathlib import Path

path = Path("pretrain_t4.py")
text = path.read_text()

if "_AMP_USE_TORCH_AMP" not in text:
    text = text.replace(
        "from torch.cuda.amp import GradScaler, autocast",
        "try:\n"
        "    from torch.amp import GradScaler, autocast\n"
        "    _AMP_DEVICE = \"cuda\"\n"
        "    _AMP_USE_TORCH_AMP = True\n"
        "except Exception:\n"
        "    from torch.cuda.amp import GradScaler, autocast\n"
        "    _AMP_DEVICE = None\n"
        "    _AMP_USE_TORCH_AMP = False",
    )

    text = text.replace(
        "    scaler = GradScaler(enabled=use_amp)",
        "    if _AMP_USE_TORCH_AMP:\n"
        "        scaler = GradScaler(_AMP_DEVICE, enabled=use_amp)\n"
        "    else:\n"
        "        scaler = GradScaler(enabled=use_amp)",
    )

    text = text.replace(
        "            with autocast(device_type=\"cuda\", enabled=use_amp, dtype=torch.float16):",
        "            if _AMP_USE_TORCH_AMP:\n"
        "                amp_ctx = autocast(_AMP_DEVICE, enabled=use_amp, dtype=torch.float16)\n"
        "            else:\n"
        "                amp_ctx = autocast(enabled=use_amp, dtype=torch.float16)\n"
        "            with amp_ctx:",
    )

    path.write_text(text)
    print("✅ Patched AMP usage in pretrain_t4.py")
else:
    print("✅ AMP patch already present")

<div style="padding: 15px 20px; background-color: #F3E5F5; border-left: 6px solid #9C27B0; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #6A1B9A; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">3️⃣</span> Build & Verify Model
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Quick sanity check to ensure the T4 config builds and runs a forward pass.</p>
</div>

In [ ]:
# ── Build model with T4-optimised config ─────────────────────────
from mytho_model import MythoConfig, MythoModel
from data import VOCAB_SIZE

config = MythoConfig(
    vocab_size=VOCAB_SIZE,   # 50257 (GPT-2 BPE)
    d_model=768,
    n_heads=12,
    d_head=64,
    d_latent_kv=256,
    d_rope=32,
    n_experts=8,
    n_active_experts=2,
    d_expert_ff=2048,
    max_depth=8,             # ← reduced for T4
    max_seq_len=512,         # ← reduced for T4
    dropout=0.0,
)

model = MythoModel(config).cuda()
n_params = model.num_parameters()
print(f"✅ Model built: {n_params:,} parameters ({n_params/1e6:.1f}M)")
print(f"   Vocab: {config.vocab_size}  |  d_model: {config.d_model}")
print(f"   Heads: {config.n_heads}  |  Experts: {config.n_experts} (top-{config.n_active_experts})")
print(f"   Max depth: {config.max_depth}  |  Seq len: {config.max_seq_len}")

# Quick forward pass test
ids = torch.randint(1, config.vocab_size, (2, 64), device="cuda")
with torch.no_grad():
    out = model(ids, labels=ids)
print(f"\n   Forward pass OK — Loss: {out['loss'].item():.4f}, Depth: {out['mean_depth'].item():.1f}")

del model, ids, out
torch.cuda.empty_cache()

<div style="padding: 15px 20px; background-color: #FBE9E7; border-left: 6px solid #FF5722; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #D84315; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">4️⃣</span> Data Pipeline
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Streams <a href='https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu' target='_blank' style='color: #E64A19;'>FineWeb-Edu</a> and tokenizes with GPT-2 BPE (tiktoken). First run downloads a small shard.</p>
</div>

In [ ]:
# ── Test tokeniser + data stream ─────────────────────────────────
from data import tokenise, decode, create_dataloader, VOCAB_SIZE

# Tokeniser roundtrip
text = "The recurrent depth transformer processes tokens iteratively."
ids = tokenise(text)
assert decode(ids) == text
print(f"✅ Tokeniser OK — {len(ids)} tokens for {len(text)} chars")

# Test dataloader (small batch)
print("\n⏳ Testing data stream (this downloads a small sample)...")
dl = create_dataloader(seq_len=128, batch_size=2, max_docs=10, num_workers=0)
batch_x, batch_y = next(iter(dl))
print(f"✅ Data OK — batch shape: {batch_x.shape}")
print(f"   Sample tokens: {batch_x[0, :20].tolist()}")
print(f"   Decoded: {decode(batch_x[0, :40].tolist())[:80]}...")
del dl, batch_x, batch_y

<div style="padding: 15px 20px; background-color: #E0F7FA; border-left: 6px solid #00BCD4; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #00838F; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">5️⃣</span> Google Drive (Optional)
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Mount Drive for persistent checkpoints, or keep everything in local Colab storage.</p>
</div>

In [ ]:
# ── Mount Google Drive (uncomment to enable) ────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CKPT_DIR = "/content/drive/MyDrive/mytho_checkpoints"

# ── Or use local storage ────────────────────────────────────────
CKPT_DIR = "checkpoints_t4"
print(f"✅ Checkpoints will save to: {CKPT_DIR}")

<div style="padding: 20px; background-color: #ECEFF1; border-left: 6px solid #607D8B; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 10px 0; color: #37474F; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em;">
        <span style="margin-right: 10px;">6️⃣</span> Pretraining
    </h2>
    <p style="margin: 0 0 15px 0; color: #546E7A; font-size: 1.05em;"><b>T4 VRAM budget (~16 GB):</b></p>
    <div style="background: white; padding: 15px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
        <table style="width: 100%; border-collapse: collapse; color: #333;">
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 8px 5px;"><b>Model params (FP16)</b></td><td style="padding: 8px 5px;">~340 MB</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 8px 5px;"><b>Optimizer states (FP32)</b></td><td style="padding: 8px 5px;">~1.4 GB</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 8px 5px;"><b>Gradients (FP16)</b></td><td style="padding: 8px 5px;">~340 MB</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 8px 5px;"><b>Activations (batch=4, depth=8)</b></td><td style="padding: 8px 5px;">~2-4 GB</td></tr>
            <tr><td style="padding: 8px 5px; color: #2E7D32;"><b>Total Estimated</b></td><td style="padding: 8px 5px; color: #2E7D32; font-weight: bold;">~5-6 GB ✅</td></tr>
        </table>
    </div>
    <div style="margin-top: 15px; padding: 10px; background: #FFF9C4; border-left: 4px solid #FBC02D; border-radius: 4px; color: #F57F17;">
        💡 <b>Tip:</b> Adjust <code>--max_steps</code> for longer runs. <code>500</code> steps ≈ 5 min, <code>5000</code> steps ≈ 45 min on T4.
    </div>
</div>

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  🏋️  RUN PRETRAINING
# ═══════════════════════════════════════════════════════════════════

# Quick smoke test (≈2 min) — change max_steps for longer training
!python pretrain_t4.py \
    --max_steps 500 \
    --batch_size 4 \
    --grad_accum 8 \
    --seq_len 512 \
    --max_depth 8 \
    --lr 3e-4 \
    --warmup_steps 50 \
    --log_every 10 \
    --save_every 250 \
    --ckpt_dir {CKPT_DIR} \
    --max_docs 5000

<div style="padding: 15px 20px; background-color: #F1F8E9; border-left: 6px solid #4CAF50; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #2E7D32; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">7️⃣</span> Training Metrics
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Reads <code>train_log.jsonl</code> from the checkpoint folder and plots curves.</p>
</div>

In [ ]:
# ── Plot training curves ─────────────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

log_path = Path(CKPT_DIR) / "train_log.jsonl"
if log_path.exists():
    records = [json.loads(l) for l in log_path.read_text().strip().split("\n") if l.strip()]
    steps  = [r["step"] for r in records]
    losses = [r["loss"] for r in records]
    ce     = [r["ce_loss"] for r in records]
    depths = [r["mean_depth"] for r in records]
    lrs    = [r["lr"] for r in records]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("Mytho T4 Pretraining", fontsize=16, fontweight="bold")

    axes[0,0].plot(steps, losses, color="#e74c3c", linewidth=1.5)
    axes[0,0].set_title("Total Loss"); axes[0,0].set_xlabel("Step"); axes[0,0].grid(alpha=0.3)

    axes[0,1].plot(steps, ce, color="#3498db", linewidth=1.5)
    axes[0,1].set_title("Cross-Entropy Loss"); axes[0,1].set_xlabel("Step"); axes[0,1].grid(alpha=0.3)

    axes[1,0].plot(steps, depths, color="#2ecc71", linewidth=1.5)
    axes[1,0].set_title("Mean ACT Depth"); axes[1,0].set_xlabel("Step"); axes[1,0].grid(alpha=0.3)

    axes[1,1].plot(steps, lrs, color="#9b59b6", linewidth=1.5)
    axes[1,1].set_title("Learning Rate"); axes[1,1].set_xlabel("Step"); axes[1,1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(CKPT_DIR) / "training_curves.png", dpi=150)
    plt.show()
    print(f"\n✅ Final loss: {losses[-1]:.4f}  |  Final depth: {depths[-1]:.1f}")
else:
    print("⚠ No training log found. Run training first.")

<div style="padding: 15px 20px; background-color: #E3F2FD; border-left: 6px solid #2196F3; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #1565C0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">8️⃣</span> Text Generation
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Load the latest checkpoint and sample a few prompts.</p>
</div>

In [ ]:
# ── Generate text from checkpoint ────────────────────────────────
import torch, glob
from pathlib import Path
from mytho_model import MythoModel
from data import tokenise, decode

# Find latest checkpoint
ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    ckpt_path = ckpts[-1]
    print(f"▸ Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cuda", weights_only=False)
    config = ckpt["config"]
    model = MythoModel(config).cuda()
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"▸ Loaded at step {ckpt['step']}  |  {model.num_parameters():,} params")

    # Generate from several prompts
    prompts = [
        "The history of artificial intelligence",
        "In mathematics, a prime number is",
        "The scientific method involves",
    ]

    for prompt in prompts:
        ids = tokenise(prompt)
        input_ids = torch.tensor([ids], device="cuda")

        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=100,
                temperature=0.8, top_k=50, top_p=0.9,
            )

        generated = decode(output[0].tolist())
        print(f"\n{'─'*60}")
        print(f"Prompt: {prompt}")
        print(f"Output: {generated}")

    del model
    torch.cuda.empty_cache()
else:
    print("⚠ No checkpoints found. Run training first.")

<div style="padding: 15px 20px; background-color: #F3E5F5; border-left: 6px solid #9C27B0; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #6A1B9A; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">9️⃣</span> Advanced Features Demo
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Quick smoke tests for scratchpad, verifier, branching, and expert metrics.</p>
</div>

In [ ]:
# ── Advanced feature demos ───────────────────────────────────────
import torch
from mytho_model import (
    MythoConfig, MythoModel,
    LatentScratchpad, VerifierHead, BranchingController,
    ExpertMetrics, DynamicExpertGrowth,
)

cfg = MythoConfig(
    d_model=256, n_heads=4, d_head=64, d_latent_kv=64,
    d_rope=16, max_depth=4, n_experts=4, n_active_experts=2,
    d_expert_ff=512, max_seq_len=64, vocab_size=1000,
)
ids = torch.randint(1, 1000, (2, 32), device="cuda")

# 1. Base model
m = MythoModel(cfg).cuda()
out = m(ids, labels=ids)
print(f"[Base]        Loss: {out['loss'].item():.4f}  Depth: {out['mean_depth'].item():.1f}")

# 2. Scratchpad + Verifier-Driven ACT
ms = MythoModel(cfg, use_scratchpad=True, d_scratch=64).cuda()
out_s = ms(ids, labels=ids)
print(f"[Scratchpad]  Loss: {out_s['loss'].item():.4f}  "
      f"Conf: {out_s.get('confidence', torch.tensor(0)).item():.4f}")

# 3. Branching recurrence
mb = MythoModel(cfg, use_scratchpad=True, d_scratch=64,
                use_branching=True, n_branches=2).cuda()
out_b = mb(ids, labels=ids)
print(f"[Branching]   Loss: {out_b['loss'].item():.4f}  Depth: {out_b['mean_depth'].item():.1f}")

# 4. Expert metrics
em = ExpertMetrics(4)
logits = torch.randn(2, 32, 4)
topk_i = logits.topk(2, dim=-1).indices
em.update(logits, topk_i)
report = em.compute(m.blocks[0].moe)
print(f"[Experts]     Entropy: {report['expert_entropy']:.3f} / "
      f"{report['max_entropy']:.3f}  Stability: {report['routing_stability']:.3f}")

# 5. Generation
gen = ms.generate(ids[:1, :8], max_new_tokens=16)
print(f"[Generate]    Input: {ids[:1,:8].shape} → Output: {gen.shape}")

print("\n✅ All advanced features OK")
del m, ms, mb
torch.cuda.empty_cache()

<div style="padding: 15px 20px; background-color: #FBE9E7; border-left: 6px solid #FF5722; border-radius: 6px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
    <h2 style="margin: 0 0 8px 0; color: #D84315; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.5em; display: flex; align-items: center;">
        <span style="margin-right: 10px;">🔟</span> Scratchpad Pretraining
    </h2>
    <p style="margin: 0; color: #555; font-size: 1.05em; line-height: 1.5;">Enable the latent scratchpad + verifier-driven ACT for richer internal reasoning. Uses about 2 GB extra VRAM.</p>
</div>

In [ ]:
# ── Pretrain with scratchpad (optional — uses more VRAM) ─────────
!python pretrain_t4.py \
     --use_scratchpad \
     --d_scratch 64 \
     --max_steps 500 \
     --batch_size 4 \
     --grad_accum 8 \
     --seq_len 512 \
     --max_depth 8 \
     --ckpt_dir checkpoints_t4_scratchpad \
     --max_docs 5000